# 00 · Setup and the go/no-go pilot

Hours 0-3 of the plan. This notebook extracts the refusal direction, builds the
concept bank, and runs the pilot that decides whether the effect is measurable
at this scale at all.

**Decision rule.** If forced-choice identification is at chance both on the
unmodified model and at full ablation, stop and switch models (or fall back to
the reduced claim). Do not spend more than three hours here.


In [ ]:
#@title Clone the repo and install dependencies { display-mode: "form" }
# Colab: paste a GitHub PAT with repo:read scope. It is used only for the clone
# and is not written to disk.
import os, subprocess, sys, getpass, pathlib

REPO   = "sagnikc395/apart-mind-digital-mind"  #@param {type:"string"}
BRANCH = "main"                                 #@param {type:"string"}
WORKDIR = "/content"

if pathlib.Path("/content").exists():
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token (blank if public): ")
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    dest = pathlib.Path(WORKDIR) / REPO.split("/")[-1]
    if dest.exists():
        subprocess.run(["git", "-C", str(dest), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, str(dest)], check=True)
    os.chdir(dest)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "torch", "transformers>=4.44", "accelerate", "datasets", "matplotlib"], check=True)
else:
    os.chdir(pathlib.Path.cwd())  # already inside the repo, e.g. running locally

sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("cwd:", os.getcwd())


In [ ]:
#@title Persist results to Drive (survives a session kill)
import os, pathlib

RESULTS = "/content/drive/MyDrive/alignment_tax_results"  #@param {type:"string"}
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    RESULTS = str(pathlib.Path.cwd() / "results")
    print("no Drive; writing to", RESULTS, f"({exc})")
os.environ["ALIGNMENT_TAX_RESULTS"] = RESULTS
pathlib.Path(RESULTS).mkdir(parents=True, exist_ok=True)
print("results ->", RESULTS)


In [ ]:
#@title Build the bundled data files (concepts, baseline corpus, offline fallbacks)
!python -m alignment_tax.build_data

In [ ]:
#@title Run configuration
from pathlib import Path
import os
from alignment_tax.config import RunConfig, LAMBDA_GRID

cfg = RunConfig()
cfg.model.name = "Qwen/Qwen3-4B-Instruct-2507"  #@param {type:"string"}
cfg.model.batch_size = 16                        #@param {type:"integer"}
cfg.lambdas = LAMBDA_GRID                        # 0, 0.25, 0.5, 0.75, 1.0
cfg.results_dir = Path(os.environ.get("ALIGNMENT_TAX_RESULTS", "results"))
cfg.save()
print(cfg.run_dir)


In [ ]:
#@title Load the model
from alignment_tax import pipeline

hm = pipeline.load_model(cfg)

## Refusal direction

Difference-in-means over harmful (AdvBench) minus harmless (Alpaca) at every
(post-instruction position, layer) pair, then selected on *held-out* HarmBench +
Alpaca by bypass score, induce score > 0, KL < 0.1 on harmless inputs, and
layer < 0.8L. The selected layer and position are reported in the paper.

In [ ]:
rd = pipeline.stage_direction(hm, cfg)
print("selected layer:", rd.layer, "position:", rd.position)
print("bypass:", round(rd.scores.bypass_score, 3),
      "induce:", round(rd.scores.induce_score, 3),
      "KL:", round(rd.scores.kl, 4))

## Concept bank

Difference-of-means concept vectors against a generic baseline corpus, unit
normalised, plus the mean residual-stream norm at each candidate injection layer
(that norm is what makes the injection strength `alpha` comparable across layers
and model families).

In [ ]:
bank = pipeline.stage_concepts(hm, cfg)
print(len(bank.names), "concepts; norm scale:", bank.norm_scale)

## Pilot

20 concepts, alpha in {2, 4}, three layers spanning 0.5-0.85 of depth, at
lambda in {0, 1}, using prefill forced identification and k-way forced choice.

In [ ]:
decision = pipeline.stage_pilot(hm, cfg, bank, rd.vector)
import pandas as pd
pd.DataFrame(decision["rows"]).sort_values("accuracy", ascending=False).head(12)

In [ ]:
#@title Fix the injection layer and alpha for the main sweep
best = decision["verdict"]["best_cell"]
if best:
    cfg.injection.layer = int(best["layer"])
    cfg.injection.alpha = float(best["alpha"])
    cfg.save()
print("locked layer:", cfg.injection.layer, "alpha:", cfg.injection.alpha)
print("decision:", decision["verdict"]["decision"])